In [1]:
import sys
sys.path.append('../../Simulate/')

import os
import random
import numpy as np
import subprocess

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple
from threading import Lock
from concurrent.futures import ThreadPoolExecutor

from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from UtilityFunctions import get_htsim_path
from ParseGenome import ParseGenome
from BSReadSim import BSReadSim

In [2]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "data/ref/BSB_test.fa"
outdir = working_path + "outdir"

In [3]:
self = BSReadSim(ref_fasta=ref_fasta, outdir=outdir, overwrite_db=True, 
                 meth_db_path='/home/wbguo/iproject/BSReadSim/test/outdir/',
                 n_threads=1, num_reads=1000, 
                 verbose =True, shuffle=False, gzip=False)

Initiating genome...
Initiating methylation profile...

[Initiating meth_db] for chr10...

[Initiating meth_db] for chr11...

[Initiating meth_db] for chr12...

[Initiating meth_db] for chr13...

[Initiating meth_db] for chr14...

[Initiating meth_db] for chr15...


../../Simulate/StreamReads.py:39: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:39: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [4]:
contig_id = 'chr10'

In [5]:
sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

In [6]:
sim_cmd

['/home/wbguo/iproject/BSReadSim/HTSIM/htsim',
 '/home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa',
 '-i',
 '400',
 '-I',
 '25',
 '-m',
 '100',
 '-M',
 '1000',
 '-1',
 '100',
 '-2',
 '100',
 '-e',
 '0',
 '-A',
 '0.05',
 '-u',
 '1',
 '-f',
 '1',
 '-g',
 'None',
 '-r',
 '0.001',
 '-R',
 '0.15',
 '-X',
 '0.15',
 '-h',
 '0',
 '-s',
 '-1',
 '-T',
 '0',
 '-x',
 'None',
 '-b',
 'None',
 '-B',
 'None',
 '-D',
 'None',
 '-c',
 'chr10',
 '-n',
 '107']

In [7]:
' '.join(sim_cmd)

'/home/wbguo/iproject/BSReadSim/HTSIM/htsim /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa -i 400 -I 25 -m 100 -M 1000 -1 100 -2 100 -e 0 -A 0.05 -u 1 -f 1 -g None -r 0.001 -R 0.15 -X 0.15 -h 0 -s -1 -T 0 -x None -b None -B None -D None -c chr10 -n 107'

In [8]:
read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end))

In [9]:
var_contig, sim_data= next(read_gen)

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1677650863
[sim_core] contig 'chr10': simulate 107 reads...


In [10]:
var_contig

'chr10'

In [11]:
self.current_contig = var_contig

In [12]:
self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)

In [13]:
len(self.pos_map)

423500

In [14]:
self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)

In [15]:
sim_data # 0-based

{296: {'chrom': 'chr10',
  'pos': 296,
  'ref': 'A',
  'alt': 'C',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('A', 'C'),
  'meth': 0.07275,
  'ctx': 1},
 637: {'chrom': 'chr10',
  'pos': 637,
  'ref': 'T',
  'alt': 'C',
  'offset': 0,
  'heter': False,
  'indel': 0,
  'iupac': ('C',),
  'meth': 0.0,
  'ctx': 7},
 1803: {'chrom': 'chr10',
  'pos': 1803,
  'ref': 'A',
  'alt': '-',
  'offset': -1,
  'heter': False,
  'indel': -1,
  'iupac': None},
 2824: {'chrom': 'chr10',
  'pos': 2824,
  'ref': 'A',
  'alt': 'T',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('A', 'T')},
 2984: {'chrom': 'chr10',
  'pos': 2984,
  'ref': 'G',
  'alt': 'A',
  'offset': 0,
  'heter': False,
  'indel': 0,
  'iupac': ('A',)},
 4438: {'chrom': 'chr10',
  'pos': 4438,
  'ref': 'T',
  'alt': 'A',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iupac': ('A', 'T')},
 4948: {'chrom': 'chr10',
  'pos': 4948,
  'ref': 'C',
  'alt': 'G',
  'offset': 0,
  'heter': True,
  'indel': 0,
  'iup

In [16]:
self.variant_profile

{296: (0.07275, 1),
 637: (0.0, 7),
 4948: (0.10046, 15),
 5032: (0.0, 7),
 7373: (0.003096, 9),
 10158: (0.0, 7),
 14590: (0.1953, 1),
 16282: (array([1.], dtype=float16), array([3], dtype=int16)),
 17074: (0.0, 3),
 18434: (0.0656, 1),
 19071: (0.0009747, 1),
 19837: (0.0, 15),
 22494: (0.0, 7),
 29209: (0.0, 11),
 30906: (0.10535, 1),
 33040: (0.889, 9),
 36625: (0.0, 15),
 40059: (0.0, 15),
 41102: (0.694, 1),
 43477: (array([6.e-08], dtype=float16), array([7], dtype=int16)),
 44179: (0.966, 1),
 48990: (0.0, 15),
 51987: (0.0, 7),
 52704: (0.0, 7),
 53604: (0.0, 15),
 56218: (array([0.], dtype=float16), array([7], dtype=int16)),
 57381: (1.0, 11),
 63024: (0.0, 15),
 66445: (0.0, 7),
 67866: (0.0, 15),
 68166: (0.0, 15),
 69262: (0.6396, 9),
 70409: (array([0.], dtype=float16), array([3], dtype=int16)),
 70841: (0.0, 3),
 74945: (array([0.], dtype=float16), array([15], dtype=int16)),
 78437: (0.0, 15),
 80695: (0.0, 15),
 82293: (0.9995, 15),
 82918: (0.0, 15),
 85722: (1.0, 11),


In [113]:
for _, read_pair in read_gen:
    if read_pair[0]['n_sub'] !=0:
        break

In [114]:
read_pair

[{'read_id': '@chr10:170749:171161:5e',
  'pair': 0,
  'strand': -1,
  'flag_pos': 1,
  'flag_mut': 65420,
  'flag_indel': 0,
  'start': 170748,
  'end': 170848,
  'cover_pos': 1,
  'n_sub': 1,
  'n_indel': 0,
  'insert_size': 412,
  'inner_dist': 212,
  'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
  'seq': array([1, 0, 0, 2, 3, 3, 3, 3, 2, 0, 0, 0, 0, 0, 1, 0, 1, 3, 3, 3, 3, 3,
         2, 3, 2, 2, 0, 0, 3, 3, 3, 2, 1, 0, 0, 2, 3, 2, 2, 0, 2, 0, 3, 3,
         3, 1, 0, 0, 2, 1, 2, 0, 0, 2, 1, 3, 1, 3, 0, 1, 3, 0, 0, 2, 0, 2,
         0, 0, 2, 0, 0, 2, 2, 0, 2, 0, 0, 3, 1, 0, 2, 0, 2, 0, 0, 3, 2, 2,
         2, 0, 3, 0, 1, 3, 3, 0, 2, 2, 2, 0], dtype=int8),
  'ofs': array(

In [115]:
len(read_pair[0]['seq'])

100

In [116]:
len(read_pair[0]['ctx'])

100

In [117]:
len(read_pair[1]['ctx'])

100

In [118]:
len(read_pair[1]['seq'])

100

In [119]:
read1_idx   = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']

In [120]:
pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx

In [121]:
read1_idx

0

In [122]:
pattern_idx

0

In [123]:
self.mask_context(read_pair[0], pattern_idx)
self.mask_context(read_pair[1], pattern_idx)

In [124]:
read_pair

[{'read_id': '@chr10:170749:171161:5e',
  'pair': 0,
  'strand': -1,
  'flag_pos': 1,
  'flag_mut': 65420,
  'flag_indel': 0,
  'start': 170748,
  'end': 170848,
  'cover_pos': 1,
  'n_sub': 1,
  'n_indel': 0,
  'insert_size': 412,
  'inner_dist': 212,
  'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
  'seq': array([1, 0, 0, 2, 3, 3, 3, 3, 2, 0, 0, 0, 0, 0, 1, 0, 1, 3, 3, 3, 3, 3,
         2, 3, 2, 2, 0, 0, 3, 3, 3, 2, 1, 0, 0, 2, 3, 2, 2, 0, 2, 0, 3, 3,
         3, 1, 0, 0, 2, 1, 2, 0, 0, 2, 1, 3, 1, 3, 0, 1, 3, 0, 0, 2, 0, 2,
         0, 0, 2, 0, 0, 2, 2, 0, 2, 0, 0, 3, 1, 0, 2, 0, 2, 0, 0, 3, 2, 2,
         2, 0, 3, 0, 1, 3, 3, 0, 2, 2, 2, 0], dtype=int8),
  'ofs': array(

In [125]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

Read1:CAAGTTTTGAAAAACACTTTTTGTGGAATTTGCAAGTGGAGATTTCAAGCGAAGCTCTACTAAGAGAAGAAGGAGAATCAGAGAATGGGATACTTAGGGA
Read2:CTTGAATTTCTCCCTAGAAAATGGTTTTTTCTTTTCTACTATCAGGGTTAAATACTCCAGCATTCCTTGGAAAACACTTCTGTGAGCATTCAGGCTTTAT


In [126]:
self.retrive_meth_db(read_pair[0])
self.retrive_meth_db(read_pair[1])

In [127]:
read_pair

[{'read_id': '@chr10:170749:171161:5e',
  'pair': 0,
  'strand': -1,
  'flag_pos': 1,
  'flag_mut': 65420,
  'flag_indel': 0,
  'start': 170748,
  'end': 170848,
  'cover_pos': 1,
  'n_sub': 1,
  'n_indel': 0,
  'insert_size': 412,
  'inner_dist': 212,
  'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
  'seq': array([1, 0, 0, 2, 3, 3, 3, 3, 2, 0, 0, 0, 0, 0, 1, 0, 1, 3, 3, 3, 3, 3,
         2, 3, 2, 2, 0, 0, 3, 3, 3, 2, 1, 0, 0, 2, 3, 2, 2, 0, 2, 0, 3, 3,
         3, 1, 0, 0, 2, 1, 2, 0, 0, 2, 1, 3, 1, 3, 0, 1, 3, 0, 0, 2, 0, 2,
         0, 0, 2, 0, 0, 2, 2, 0, 2, 0, 0, 3, 1, 0, 2, 0, 2, 0, 0, 3, 2, 2,
         2, 0, 3, 0, 1, 3, 3, 0, 2, 2, 2, 0], dtype=int8),
  'ofs': array(

In [130]:
self.set_context_state(read_pair)

In [131]:
read_pair

[{'read_id': '@chr10:170749:171161:5e',
  'pair': 0,
  'strand': -1,
  'flag_pos': 1,
  'flag_mut': 65420,
  'flag_indel': 0,
  'start': 170748,
  'end': 170848,
  'cover_pos': 1,
  'n_sub': 1,
  'n_indel': 0,
  'insert_size': 412,
  'inner_dist': 212,
  'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
  'seq': array([1, 0, 0, 2, 3, 3, 3, 3, 2, 0, 0, 0, 0, 0, 1, 0, 1, 3, 3, 3, 3, 3,
         2, 3, 2, 2, 0, 0, 3, 3, 3, 2, 1, 0, 0, 2, 3, 2, 2, 0, 2, 0, 3, 3,
         3, 1, 0, 0, 2, 1, 2, 0, 0, 2, 1, 3, 1, 3, 0, 1, 3, 0, 0, 2, 0, 2,
         0, 0, 2, 0, 0, 2, 2, 0, 2, 0, 0, 3, 1, 0, 2, 0, 2, 0, 0, 3, 2, 2,
         2, 0, 3, 0, 1, 3, 3, 0, 2, 2, 2, 0], dtype=int8),
  'ofs': array(

In [87]:
read_rec = read_pair[1]

In [88]:
unmeth_idx = np.squeeze(np.where(np.bitwise_and(read_rec['ctx'], 0x1)==1)) 

In [89]:
unmeth_idx

array([ 1,  7, 19, 32, 43, 54, 70, 72, 75, 77, 83, 91, 92])

In [90]:
conv_states= bernoulli.rvs(self.conversion_rate, size=len(unmeth_idx))

In [91]:
conv_states

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [92]:
conv_idx   = unmeth_idx[conv_states == 1]

In [93]:
conv_idx

array([ 1,  7, 19, 32, 43, 54, 70, 72, 75, 77, 83, 91, 92])

In [94]:
read_rec['seq']

array([0, 1, 3, 0, 3, 0, 2, 1, 2, 3, 2, 2, 3, 3, 3, 3, 0, 2, 2, 1, 0, 3,
       3, 3, 3, 0, 2, 2, 2, 3, 2, 0, 1, 0, 3, 2, 2, 2, 0, 0, 2, 2, 3, 1,
       0, 0, 2, 3, 2, 0, 3, 3, 0, 3, 1, 3, 3, 2, 2, 0, 0, 0, 3, 3, 0, 3,
       3, 3, 2, 2, 1, 3, 1, 0, 3, 1, 3, 1, 0, 0, 0, 1, 3, 1, 3, 3, 2, 3,
       3, 2, 2, 1, 1, 0, 2, 2, 1, 3, 2, 2], dtype=int8)

In [95]:
read_rec['seq'][conv_idx] = np.bitwise_and(read_rec['seq'][conv_idx] + 2, 0x3) 

In [96]:
read_rec['seq']

array([0, 3, 3, 0, 3, 0, 2, 3, 2, 3, 2, 2, 3, 3, 3, 3, 0, 2, 2, 3, 0, 3,
       3, 3, 3, 0, 2, 2, 2, 3, 2, 0, 3, 0, 3, 2, 2, 2, 0, 0, 2, 2, 3, 3,
       0, 0, 2, 3, 2, 0, 3, 3, 0, 3, 3, 3, 3, 2, 2, 0, 0, 0, 3, 3, 0, 3,
       3, 3, 2, 2, 3, 3, 3, 0, 3, 3, 3, 3, 0, 0, 0, 1, 3, 3, 3, 3, 2, 3,
       3, 2, 2, 3, 3, 0, 2, 2, 1, 3, 2, 2], dtype=int8)

In [ ]:
unconv_idx = unmeth_idx[conv_states == 0]

In [34]:
self.treat_bisulfite(read_pair[0])
self.treat_bisulfite(read_pair[1])

In [35]:
read_pair

[{'read_id': '@chr10:288254:288611:a',
  'pair': 0,
  'strand': -1,
  'flag_pos': 1,
  'flag_mut': 65420,
  'flag_indel': 0,
  'start': 288253,
  'end': 288353,
  'cover_pos': 1,
  'n_sub': 1,
  'n_indel': 0,
  'insert_size': 357,
  'inner_dist': 157,
  'cgr': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8),
  'seq': array([1, 3, 0, 1, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 3, 0, 1,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 1, 3, 0, 0, 0, 1, 0,
         1, 0, 0, 0, 2, 1, 0, 1, 3, 1, 3, 0, 0, 0, 0, 2, 0, 1, 1, 0, 0, 2,
         0, 1, 0, 0, 0, 3, 0, 0, 0, 3, 1, 0, 3, 3, 3, 0, 0, 0, 0, 3, 1, 0,
         0, 0, 0, 0, 3, 3, 1, 0, 0, 0, 2, 0], dtype=int8),
  'ofs': array([

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

In [ ]:
self.rev_complement(read_pair)

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

In [ ]:
self.add_seq_err(read_pair[0])
self.add_seq_err(read_pair[1])

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

In [ ]:
self.add_qual_score(read_pair[0])
self.add_qual_score(read_pair[1])

In [ ]:
read_pair

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq

In [ ]:
x = Seq("TCATGACACGGAACACAAACTGCAACCTCTGCCTCACGGGTTCAAATGATTCTCCTGCCTCAGCCTTCTGGTAAATGTCTAGATGGTGAGGATTAGGTTA")

In [ ]:
str(x.reverse_complement())

In [ ]:
read_rec = read_pair[0]

In [ ]:
unmeth_idx = np.squeeze(np.where(np.bitwise_and(read_rec['ctx'], 0x1)==1)) # behave strange without ==1

In [ ]:
unmeth_idx

In [ ]:
conv_states= bernoulli.rvs(self.conversion_rate, size=len(unmeth_idx))

In [ ]:
conv_states == 0

In [ ]:
conv_idx   = unmeth_idx[np.where(conv_states)]

In [ ]:
read_rec['seq'][conv_idx]  = np.bitwise_and(read_rec['seq'][conv_idx] + 2, 0x3) 

In [ ]:
read_rec['seq'][conv_idx]

In [ ]:
conv_base  = np.bitwise_and(read_rec['seq'][conv_idx] + 2, 0x3) # C2T, G2A


In [ ]:
np.place(read_rec['seq'], conv_idx, conv_base)

In [ ]:
type(read_rec['seq'])

In [ ]:
read_rec = read_pair
read1_state = self.fetch_meth_state(read_rec[0]['meth'])

In [ ]:
read_meth = read_rec[0]['meth']

In [ ]:
meth_states = np.zeros(len(read_meth))

In [ ]:
meth_states
nonzero_idx = np.where(read_meth)[0]

In [ ]:
meth_states[nonzero_idx] = bernoulli.rvs(read_meth[nonzero_idx], size = len(nonzero_idx))

In [ ]:
meth_states